# PIPELINE LENGKAP: ANOTASI GEMINI + TRAINING INDOBERT + INFERENSI + SNA (LII)
============================================================================
**ALL-IN-ONE Google Colab Notebook dengan GPU T4**  
**Proyek:** Analisis Sentimen, 9 Emosi Plutchik & Social Network Analysis (SNA) Opini Publik Lembaga Islam Indonesia (LII)  
**Topik Utama:** `Nahdlatul_Ulama` (NU), `Muhammadiyah`, & `BAZNAS`

## CARA PAKAI (STEP-BY-STEP)
1. Buka **Google Drive** (akun yang sama dengan Google Colab).
2. Buat folder baru di `MyDrive` / `Drive Saya` dengan nama: **`IndoBERT_SNA_LII`**
3. Upload file dari folder lokal `IndoBERT_SNA_MT_LII/data/` ke dalam `MyDrive/IndoBERT_SNA_LII/`:
   - **Wajib:**
     - `master_scraped_corpus.parquet` *(138.756 baris korpus bersih 7 platform — atau upload `master_scraped_corpus.csv`)*
   - **Sangat Disarankan / Opsional (Sudah tersedia di `data/` lokal untuk mempercepat):**
     - `gold_standard_annotated_1000.csv` *(jika 1.000 baris sudah lengkap, Cell 3 otomatis melewati panggilan Gemini API)*
     - `gold_standard_sample_1000.csv` *(opsional jika ingin annotasi ulang di Colab)*
     - `train.csv`, `val.csv`, `test.csv` *(dataset latih 3-Sentimen yang sudah disiapkan)*
     - `train_emotion_9.csv`, `val_emotion_9.csv`, `test_emotion_9.csv` *(dataset latih 9-Emosi)*
   - **Keamanan API Key Gemini (Jika ingin menjalankan Cell 3 Gemini API di Colab):**
     - Tambahkan `GEMINI_API_KEY` *(dan opsional `GEMINI_API_KEY_2` s.d. `GEMINI_API_KEY_10`)* melalui **Colab Secrets** (ikon kunci di panel kiri Colab) **ATAU** upload file `.env` ke `MyDrive/IndoBERT_SNA_LII/.env`.
     - **Jangan** menempelkan API key langsung di dalam kode cell notebook.
4. Buka notebook ini di Google Colab: `File -> Upload notebook -> pilih IndoBERT_SNA_Training_Pipeline_LII.ipynb`
5. Aktifkan GPU T4: `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save`
6. Jalankan seluruh pipeline: `Runtime -> Run all` (`Ctrl + F9`)
7. Setelah selesai, ekstrak file `hasil_training_indobert_sna_LII.zip` ke project lokal:
   - `best_indobert_sentiment/` -> `models/best_indobert_sentiment/`
   - `best_indobert_emotion/` -> `models/best_indobert_emotion/`
   - `enriched_corpus_for_sna.*` -> `data/` *(opsional jika ingin langsung pakai hasil inferensi Colab)*
   - `sna_output/*` -> `sna/output/` *(opsional; atau jalankan `python -u run_pipeline.py infer` dan `python -u run_pipeline.py sna` secara lokal)*
============================================================================

## CELL 1: INSTALL & SETUP

In [ ]:
!pip install -q transformers torch accelerate scikit-learn seaborn pandas pyarrow google-genai networkx python-dotenv

import os
import sys
import re
import time
import json
import glob
import random
import shutil
import warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_cuda = (device.type == 'cuda')
print(f"Device Aktif : {device}")
if use_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM   : {props.total_memory / 1e9:.2f} GB")
else:
    print("PERINGATAN: GPU CUDA tidak aktif! Aktifkan lewat Runtime -> Change runtime type -> T4 GPU.")

## CELL 2: MOUNT DRIVE, CONFIG LII & SHARED HELPERS

In [ ]:
# pyright: reportMissingImports=false
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/IndoBERT_SNA_LII'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Folder Kerja Google Drive: {DRIVE_DIR}")

# === KONFIGURASI PROYEK LII ===
LII_TOPICS = ['Nahdlatul_Ulama', 'Muhammadiyah', 'BAZNAS']
MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LENGTH = 128
BATCH_SIZE = 16
SENT_EPOCHS = 3
EMO_EPOCHS = 4
LR = 2e-5

# Toggle kontrol eksekusi ulang
SKIP_IF_MODEL_EXISTS = False   # False = selalu latih ulang model IndoBERT fresh
FORCE_INFER = True             # True = selalu jalankan inferensi ulang di seluruh korpus

# === PATH FILE DI GOOGLE DRIVE ===
MASTER_PQ_PATH = os.path.join(DRIVE_DIR, 'master_scraped_corpus.parquet')
MASTER_CSV_PATH = os.path.join(DRIVE_DIR, 'master_scraped_corpus.csv')
ENRICHED_PQ_PATH = os.path.join(DRIVE_DIR, 'enriched_corpus_for_sna.parquet')
ENRICHED_CSV_PATH = os.path.join(DRIVE_DIR, 'enriched_corpus_for_sna.csv')
GOLD_SAMPLE_PATH = os.path.join(DRIVE_DIR, 'gold_standard_sample_1000.csv')
GOLD_ANNOTATED_PATH = os.path.join(DRIVE_DIR, 'gold_standard_annotated_1000.csv')
ANNOTATION_CKPT_PATH = os.path.join(DRIVE_DIR, 'annotation_checkpoint.json')

SENT_DIR = os.path.join(DRIVE_DIR, 'best_indobert_sentiment')
EMO_DIR = os.path.join(DRIVE_DIR, 'best_indobert_emotion')
SNA_OUT_DIR = os.path.join(DRIVE_DIR, 'sna_output')
os.makedirs(SENT_DIR, exist_ok=True)
os.makedirs(EMO_DIR, exist_ok=True)
os.makedirs(SNA_OUT_DIR, exist_ok=True)

# === TAKSONOMI LABEL LII ===
SENTIMENT_LABELS = ['Negatif', 'Netral', 'Positif']
SENT2ID = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
ID2SENT = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

EMOTION_9_CLASSES = [
    'Trust',         # 0: Positif
    'Anticipation',  # 1: Positif
    'Joy',           # 2: Positif
    'Disgust',       # 3: Negatif
    'Fear',          # 4: Negatif
    'Sadness',       # 5: Negatif
    'Anger',         # 6: Negatif
    'Surprise',      # 7: Netral
    'Neutral',       # 8: Netral
]
EMO2ID = {name: idx for idx, name in enumerate(EMOTION_9_CLASSES)}
ID2EMO = {idx: name for idx, name in enumerate(EMOTION_9_CLASSES)}
EMOTION_TO_SENTIMENT = {
    'Trust': 'Positif', 'Anticipation': 'Positif', 'Joy': 'Positif',
    'Disgust': 'Negatif', 'Fear': 'Negatif', 'Sadness': 'Negatif', 'Anger': 'Negatif',
    'Surprise': 'Netral', 'Neutral': 'Netral',
}
STANDARDIZE_EMO_MAP = {
    'Senang': 'Joy', 'Joy': 'Joy',
    'Percaya': 'Trust', 'Trust': 'Trust',
    'Antisipasi': 'Anticipation', 'Anticipation': 'Anticipation',
    'Jijik': 'Disgust', 'Disgust': 'Disgust',
    'Takut': 'Fear', 'Fear': 'Fear',
    'Sedih': 'Sadness', 'Sadness': 'Sadness',
    'Marah': 'Anger', 'Anger': 'Anger',
    'Terkejut': 'Surprise', 'Surprise': 'Surprise',
    'Informatif': 'Neutral', 'Netral': 'Neutral', 'Neutral': 'Neutral',
}
LEGACY_TOPIC_MAP = {
    'Nahdlatul_Ulama': 'Nahdlatul_Ulama',
    'Banom_dan_Warga_NU': 'Nahdlatul_Ulama',
    'Program_dan_Ekonomi_NU': 'Nahdlatul_Ulama',
    'Muhammadiyah': 'Muhammadiyah',
    'BAZNAS': 'BAZNAS',
    'Kelembagaan_BAZNAS': 'BAZNAS',
    'Zakat_dan_Program_BAZNAS': 'BAZNAS',
    'Layanan_dan_Akuntabilitas_BAZNAS': 'BAZNAS',
}


def load_gemini_api_keys():
    """Muat daftar Gemini API key secara aman dari .env di Drive, Colab Secrets, atau env var."""
    key_names = ['GEMINI_API_KEY'] + [f'GEMINI_API_KEY_{i}' for i in range(2, 16)] + ['GOOGLE_API_KEY']
    candidates = []

    env_path = os.path.join(DRIVE_DIR, '.env')
    if os.path.exists(env_path):
        try:
            from dotenv import dotenv_values
            env_vals = dotenv_values(env_path)
            for k in key_names:
                val = str(env_vals.get(k) or '').strip()
                if val:
                    candidates.append(val)
        except Exception as e:
            print(f"Catatan saat membaca {env_path}: {e}")

    try:
        from google.colab import userdata
        for k in key_names:
            try:
                val = str(userdata.get(k) or '').strip()
                if val:
                    candidates.append(val)
            except Exception:
                pass
    except Exception:
        pass

    for k in key_names:
        val = str(os.environ.get(k) or '').strip()
        if val:
            candidates.append(val)

    deduped = []
    seen = set()
    for val in candidates:
        if val and val not in seen:
            seen.add(val)
            deduped.append(val)
    return deduped


def normalize_indobert_text(text):
    """Ekspansi reduplikasi angka 2 / superscript 2 agar selaras dengan src/indobert_engine.py."""
    t = str(text or '').strip()
    t = re.sub(r'([a-zA-Z]+)[²2]', r'\1-\1', t)
    return t


def normalize_lii_topics(df):
    raw_t = df['topic'].fillna('').astype(str).str.strip() if 'topic' in df.columns else pd.Series('', index=df.index)
    mapped = raw_t.map(LEGACY_TOPIC_MAP)
    if mapped.isna().any():
        kw_s = df['keyword'].fillna('').astype(str).str.lower() if 'keyword' in df.columns else pd.Series('', index=df.index)
        txt_s = df['cleaned_content'].fillna('').astype(str).str.lower() if 'cleaned_content' in df.columns else pd.Series('', index=df.index)
        comb = kw_s + ' ' + txt_s
        fallback = pd.Series('Nahdlatul_Ulama', index=df.index)
        fallback = fallback.mask(comb.str.contains(r'muhammadiyah|aisyiyah|lazismu|amal usaha', na=False), 'Muhammadiyah')
        fallback = fallback.mask(comb.str.contains(r'baznas|baznaz|amil zakat', na=False), 'BAZNAS')
        mapped = mapped.fillna(fallback)
    return mapped


def find_lii_file(filename):
    """Cari file di DRIVE_DIR, subfolder data/, folder LII lain, /content, atau kedalaman 3 MyDrive."""
    candidates = [
        os.path.join(DRIVE_DIR, filename),
        os.path.join(DRIVE_DIR, 'data', filename),
        f'/content/drive/MyDrive/IndoBERT_SNA_MT_LII/data/{filename}',
        f'/content/drive/MyDrive/IndoBERT_SNA_MT_LII/{filename}',
        f'/content/drive/MyDrive/{filename}',
        f'/content/{filename}',
        f'/content/data/{filename}',
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    mydrive_root = '/content/drive/MyDrive'
    if os.path.exists(mydrive_root):
        base_depth = mydrive_root.rstrip('/').count('/')
        for root, dirs, fnames in os.walk(mydrive_root):
            if filename in fnames:
                return os.path.join(root, filename)
            if root.rstrip('/').count('/') - base_depth >= 2:
                dirs[:] = []
    return None


def load_master_corpus():
    global MASTER_PQ_PATH, MASTER_CSV_PATH
    found_pq = find_lii_file('master_scraped_corpus.parquet')
    found_csv = find_lii_file('master_scraped_corpus.csv')
    if found_pq:
        MASTER_PQ_PATH = found_pq
        print(f"Memuat korpus dari Parquet: {MASTER_PQ_PATH}")
        df = pd.read_parquet(MASTER_PQ_PATH)
    elif found_csv:
        MASTER_CSV_PATH = found_csv
        print(f"Memuat korpus dari CSV: {MASTER_CSV_PATH}")
        df = pd.read_csv(MASTER_CSV_PATH, low_memory=False)
    else:
        isi = sorted(os.listdir(DRIVE_DIR)) if os.path.exists(DRIVE_DIR) else []
        raise FileNotFoundError(
            f"master_scraped_corpus.parquet / .csv belum ditemukan di {DRIVE_DIR} (isi: {isi}). "
            "Upload terlebih dahulu file data/master_scraped_corpus.parquet ke Google Drive MyDrive/IndoBERT_SNA_LII/."
        )
    if 'content' in df.columns and 'cleaned_content' not in df.columns:
        df['cleaned_content'] = df['content']
    df['cleaned_content'] = df['cleaned_content'].fillna('').astype(str)
    df = df[df['cleaned_content'].str.strip().str.len() > 8].reset_index(drop=True)
    df['topic'] = normalize_lii_topics(df)
    return df


# === KAMUS & HEURISTIK FALLBACK LII (SELARAS DENGAN PROJECT LOKAL) ===
POSITIVE_LEXICON = {
    'bagus', 'hebat', 'mantap', 'keren', 'berhasil', 'sukses', 'bermanfaat', 'membantu',
    'apresiasi', 'dukung', 'mendukung', 'terima kasih', 'alhamdulillah', 'bersyukur', 'berkah',
    'berkualitas', 'amanah', 'bangga', 'solid', 'kompak', 'khidmat', 'peduli', 'santun',
    'damai', 'sejuk', 'rukun', 'inspiratif', 'maju', 'tumbuh', 'harapan', 'terbukti', 'layak',
    'adil', 'merata', 'beres', 'tuntas', 'positif', 'puas', 'juara', 'menyejahterakan',
    'terobosan', 'efisien', 'transparan', 'semangat', 'berdaya', 'maslahat', 'istiqomah',
}
NEGATIVE_LEXICON = {
    'buruk', 'gagal', 'rusak', 'ambrol', 'hancur', 'kecewa', 'rugi', 'korupsi',
    'bancakan', 'curang', 'bohong', 'palsu', 'maling', 'konflik', 'pecah', 'ricuh',
    'kisruh', 'polemik', 'kontroversi', 'parah', 'krisis', 'marah', 'sesat', 'bobrok',
    'bodoh', 'sengsara', 'beban', 'masalah', 'cacat', 'pencitraan', 'rugikan',
    'penipuan', 'manipulasi', 'menolak', 'tolak', 'kritik', 'protes', 'salah',
    'amburadul', 'merusak', 'gaduh', 'bubar', 'bubarkan', 'penyelewengan', 'politisasi',
}
NEGATION_WORDS = {'tidak', 'tak', 'bukan', 'jangan', 'belum', 'gak', 'ga', 'nggak', 'tdk', 'kurang'}
PROFANITY_WORDS = {
    'ajg', 'anjg', 'anjing', 'bangsat', 'bajingan', 'brengsek', 'tolol', 'bodoh',
    'goblok', 'goblog', 'idiot', 'bego', 'dungu', 'keparat', 'kampret', 'jancok',
    'jancuk', 'asu', 'bgst', 'biadab', 'bedebah', 'sialan',
}
EMOTION_LEXICON = {
    'Joy': ['senang', 'bahagia', 'gembira', 'sukacita', 'ceria', 'riang', 'puas',
            'bangga', 'syukur', 'bersyukur', 'mantap', 'keren', 'hebat', 'luar biasa', 'alhamdulillah', 'berkah'],
    'Trust': ['percaya', 'amanah', 'dukung', 'mendukung', 'apresiasi', 'salut', 'terbaik',
              'istiqomah', 'bermanfaat', 'membantu', 'peduli', 'khidmat', 'solid', 'kompak'],
    'Anticipation': ['semoga', 'berharap', 'harapan', 'rencana', 'target', 'menanti',
                     'persiapan', 'kedepan', 'masa depan', 'program', 'agenda'],
    'Sadness': ['sedih', 'duka', 'pilu', 'haru', 'menyesal', 'kecewa', 'kehilangan',
                'tragis', 'miris', 'prihatin', 'kasihan', 'berduka', 'wafat'],
    'Anger': ['marah', 'kesal', 'geram', 'murka', 'emosi', 'benci', 'muak',
              'kritik', 'protes', 'tolak', 'menolak', 'konflik', 'pecah', 'gagal', 'parah'],
    'Fear': ['takut', 'khawatir', 'cemas', 'was-was', 'ngeri', 'panik',
             'gelisah', 'ragu', 'ancaman', 'bahaya', 'waspada', 'rawan'],
    'Surprise': ['terkejut', 'kaget', 'heran', 'tercengang', 'wow', 'astaga',
                 'waduh', 'ternyata', 'mendadak', 'tiba-tiba', 'heboh', 'viral'],
    'Disgust': ['jijik', 'busuk', 'kotor', 'sampah', 'korupsi', 'bobrok',
                'memalukan', 'najis', 'hina', 'curang', 'culas', 'munafik'],
}


def rule_based_sentiment_scorer_lii(text):
    """Skor sentimen fallback kontekstual LII: 0 (Negatif), 1 (Netral), 2 (Positif)."""
    t_norm = normalize_indobert_text(text)
    words = re.findall(r'\b\w+\b', t_norm.lower())
    if not words:
        return 1
    if any(w in PROFANITY_WORDS for w in words):
        return 0
    pos_score = 0.0
    neg_score = 0.0
    for i, w in enumerate(words):
        is_neg = (i > 0 and words[i - 1] in NEGATION_WORDS) or (i > 1 and words[i - 2] in NEGATION_WORDS)
        if w in POSITIVE_LEXICON:
            if is_neg:
                neg_score += 1.5
            else:
                pos_score += 1.0
        elif w in NEGATIVE_LEXICON:
            if is_neg:
                pos_score += 0.8
            else:
                neg_score += 1.5
    diff = pos_score - neg_score
    if diff >= 1.0:
        return 2
    if diff <= -1.0:
        return 0
    return 1


def detect_emotion_lii(text):
    if not isinstance(text, str) or not text.strip():
        return 'Neutral'
    t_low = text.lower()
    scores = {emo: sum(1 for kw in kws if kw in t_low) for emo, kws in EMOTION_LEXICON.items()}
    scores = {k: v for k, v in scores.items() if v > 0}
    return max(scores, key=scores.get) if scores else 'Neutral'


def align_emotion_valence_lii(text, predicted_sentiment, raw_emotion):
    sent = str(predicted_sentiment or '').strip()
    raw_emo = STANDARDIZE_EMO_MAP.get(str(raw_emotion or '').strip(), str(raw_emotion or 'Neutral').strip())
    t_low = normalize_indobert_text(text).lower()
    if sent == 'Negatif':
        if any(w in t_low for w in ['korupsi', 'jijik', 'busuk', 'memalukan', 'curang', 'munafik']):
            return 'Disgust'
        if any(w in t_low for w in ['takut', 'khawatir', 'ancaman', 'bahaya', 'cemas']):
            return 'Fear'
        if any(w in t_low for w in ['sedih', 'duka', 'miris', 'kasihan', 'wafat', 'kecewa']):
            return 'Sadness' if raw_emo in ('Joy', 'Trust', 'Anticipation', 'Neutral') else raw_emo
        if raw_emo in ('Joy', 'Trust', 'Anticipation', 'Neutral'):
            return 'Anger'
        return raw_emo
    if sent == 'Positif':
        if raw_emo in ('Anger', 'Disgust', 'Sadness', 'Fear'):
            if any(w in t_low for w in ['dukung', 'percaya', 'amanah', 'kawal', 'istiqomah', 'bermanfaat']):
                return 'Trust'
            return 'Joy'
        return raw_emo
    if sent == 'Netral':
        if any(w in t_low for w in ['semoga', 'berharap', 'agar tidak', 'diperketat', 'ke depan']):
            return 'Anticipation'
    return raw_emo if raw_emo in EMO2ID else 'Neutral'


# === PYTORCH DATASET & TRAINER BERSAMA (REUSABLE UNTUK SENTIMEN & EMOSI) ===
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix


class IndoBERTTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(x) for x in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: self.encodings[k][idx] for k in self.encodings}
        item['labels'] = self.labels[idx]
        return item


def train_and_evaluate_indobert(
    train_df,
    val_df,
    test_df,
    label_names,
    save_dir,
    epochs=3,
    batch_size=16,
    lr=2e-5,
    max_length=128,
):
    num_labels = len(label_names)
    id2label = {i: lbl for i, lbl in enumerate(label_names)}
    label2id = {lbl: i for i, lbl in enumerate(label_names)}

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def _make_loader(df_part, shuffle=False):
        texts = [normalize_indobert_text(t) for t in df_part['text'].astype(str).tolist()]
        enc = tokenizer(texts, truncation=True, padding=False, max_length=max_length)
        ds = IndoBERTTextDataset(enc, df_part['label'].astype(int).tolist())
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collator)

    train_loader = _make_loader(train_df, shuffle=True)
    val_loader = _make_loader(val_df, shuffle=False)
    test_loader = _make_loader(test_df, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(1, len(train_loader) * epochs)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(total_steps * 0.1)), num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda)

    best_val_f1 = -1.0
    print(f"Memulai Fine-Tuning {MODEL_NAME} ({num_labels} kelas) -> {save_dir}")

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader, 1):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            with torch.amp.autocast('cuda', enabled=use_cuda):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += float(loss.item())
            if step % 150 == 0 or step == len(train_loader):
                print(f"  Epoch {epoch+1}/{epochs} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

        model.eval()
        val_preds, val_trues, val_loss = [], [], 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                with torch.amp.autocast('cuda', enabled=use_cuda):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                val_loss += float(outputs.loss.item())
                val_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                val_trues.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_trues, val_preds)
        _, _, val_f1, _ = precision_recall_fscore_support(val_trues, val_preds, average='macro', zero_division=0)
        print(
            f"  >> Epoch {epoch+1} | TrLoss={total_loss/max(1, len(train_loader)):.4f} | "
            f"ValLoss={val_loss/max(1, len(val_loader)):.4f} | ValAcc={val_acc*100:.2f}% | "
            f"ValMacroF1={val_f1*100:.2f}% ({time.time()-t0:.1f}s)"
        )
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            os.makedirs(save_dir, exist_ok=True)
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)
            print(f"  [BEST CHECKPOINT] Disimpan ke {save_dir} (ValMacroF1={val_f1*100:.2f}%)")

    best_model = AutoModelForSequenceClassification.from_pretrained(save_dir).to(device)
    best_model.eval()
    test_preds, test_trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            with torch.amp.autocast('cuda', enabled=use_cuda):
                outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
            test_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            test_trues.extend(batch['labels'].numpy())

    acc = accuracy_score(test_trues, test_preds)
    prec, rec, mf1, _ = precision_recall_fscore_support(test_trues, test_preds, average='macro', zero_division=0)
    _, _, wf1, _ = precision_recall_fscore_support(test_trues, test_preds, average='weighted', zero_division=0)
    cm = confusion_matrix(test_trues, test_preds, labels=list(range(num_labels)))
    print(f"\nHASIL TEST SET ({num_labels} KELAS): Acc={acc*100:.2f}% | Macro-F1={mf1*100:.2f}% | Weighted-F1={wf1*100:.2f}%")
    print(classification_report(test_trues, test_preds, labels=list(range(num_labels)), target_names=label_names, zero_division=0))

    return {
        'acc': float(acc), 'prec': float(prec), 'rec': float(rec),
        'mf1': float(mf1), 'wf1': float(wf1), 'cm': cm,
        'test_preds': test_preds, 'test_trues': test_trues,
        'id2label': id2label, 'label2id': label2id,
    }


GEMINI_API_KEYS = load_gemini_api_keys()
print(f"Status Gemini API Key terdeteksi: {len(GEMINI_API_KEYS)} key(s) aktif (aman, tidak dicetak).")
print("Setup Config LII selesai.")

## CELL 3: GEMINI GOLD STANDARD SAMPLING & ANNOTATION (Langkah 1)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 1: GOLD STANDARD 1.000 SAMPEL & ANOTASI GEMINI (LII)")
print("=" * 70)

def _find_lii_file(filename):
    """Cari file di DRIVE_DIR, subfolder data/, folder LII lain, /content, atau kedalaman 3 MyDrive."""
    candidates = [
        os.path.join(DRIVE_DIR, filename),
        os.path.join(DRIVE_DIR, 'data', filename),
        f'/content/drive/MyDrive/IndoBERT_SNA_MT_LII/data/{filename}',
        f'/content/drive/MyDrive/IndoBERT_SNA_MT_LII/{filename}',
        f'/content/drive/MyDrive/{filename}',
        f'/content/{filename}',
        f'/content/data/{filename}',
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    mydrive_root = '/content/drive/MyDrive'
    if os.path.exists(mydrive_root):
        base_depth = mydrive_root.rstrip('/').count('/')
        for root, dirs, fnames in os.walk(mydrive_root):
            if filename in fnames:
                return os.path.join(root, filename)
            if root.rstrip('/').count('/') - base_depth >= 2:
                dirs[:] = []
    return None


def _load_master_safe():
    global MASTER_PQ_PATH, MASTER_CSV_PATH
    found_pq = _find_lii_file('master_scraped_corpus.parquet')
    found_csv = _find_lii_file('master_scraped_corpus.csv')
    if found_pq:
        MASTER_PQ_PATH = found_pq
        print(f"Memuat korpus dari Parquet: {MASTER_PQ_PATH}")
        df = pd.read_parquet(MASTER_PQ_PATH)
    elif found_csv:
        MASTER_CSV_PATH = found_csv
        print(f"Memuat korpus dari CSV: {MASTER_CSV_PATH}")
        df = pd.read_csv(MASTER_CSV_PATH, low_memory=False)
    else:
        return None
    if 'content' in df.columns and 'cleaned_content' not in df.columns:
        df['cleaned_content'] = df['content']
    df['cleaned_content'] = df['cleaned_content'].fillna('').astype(str)
    df = df[df['cleaned_content'].str.strip().str.len() > 8].reset_index(drop=True)
    df['topic'] = normalize_lii_topics(df)
    return df


# Update fungsi global agar Cell 5, 6, 7 ikut memakai lokasi file yang ditemukan
find_lii_file = _find_lii_file

def load_master_corpus():
    df = _load_master_safe()
    if df is None:
        isi = sorted(os.listdir(DRIVE_DIR)) if os.path.exists(DRIVE_DIR) else []
        raise FileNotFoundError(
            f"master_scraped_corpus.parquet / .csv belum ditemukan! Isi {DRIVE_DIR}: {isi}. "
            "Upload data/master_scraped_corpus.parquet ke MyDrive/IndoBERT_SNA_LII/."
        )
    return df


found_gold_ann = _find_lii_file('gold_standard_annotated_1000.csv')
if found_gold_ann:
    GOLD_ANNOTATED_PATH = found_gold_ann
found_gold_smp = _find_lii_file('gold_standard_sample_1000.csv')
if found_gold_smp:
    GOLD_SAMPLE_PATH = found_gold_smp

master_raw_df = _load_master_safe()
if master_raw_df is None:
    master_raw_df = pd.DataFrame()
    isi_drive = sorted(os.listdir(DRIVE_DIR)) if os.path.exists(DRIVE_DIR) else []
    print(f"Catatan: master_scraped_corpus.parquet/.csv belum terlihat di {DRIVE_DIR} (isi: {isi_drive}).")

gold_df = pd.DataFrame()

if os.path.exists(GOLD_ANNOTATED_PATH):
    existing_gold = pd.read_csv(GOLD_ANNOTATED_PATH, low_memory=False)
    if len(existing_gold) >= 950 and 'gemini_sentiment' in existing_gold.columns:
        gold_df = existing_gold
        print(f"File Gold Standard sudah lengkap ({len(gold_df):,} baris): {GOLD_ANNOTATED_PATH}")
        print("Anotasi Gemini API dilewati otomatis agar hemat kuota.")

if gold_df.empty:
    if os.path.exists(GOLD_SAMPLE_PATH):
        sample_df = pd.read_csv(GOLD_SAMPLE_PATH, low_memory=False)
        print(f"Memuat sampel Gold Standard: {GOLD_SAMPLE_PATH} ({len(sample_df):,} baris)")
    else:
        if master_raw_df.empty:
            mydrive_folders = []
            if os.path.exists('/content/drive/MyDrive'):
                mydrive_folders = sorted([d for d in os.listdir('/content/drive/MyDrive') if 'indo' in d.lower() or 'lii' in d.lower()])
            raise FileNotFoundError(
                f"Belum ada gold_standard_annotated_1000.csv, gold_standard_sample_1000.csv, maupun master_scraped_corpus.parquet di {DRIVE_DIR}!\n"
                f"Folder terkait yang ditemukan di MyDrive: {mydrive_folders}\n"
                "Solusi: Upload file dari folder lokal data/ ke Google Drive -> MyDrive/IndoBERT_SNA_LII/, lalu jalankan ulang Cell 3."
            )
        print("Membuat 1.000 sampel terstratifikasi (7 platform x 3 topik LII) dari korpus...")
        df_s = master_raw_df.copy()
        strat_col = df_s['platform'].astype(str) + '_' + df_s['topic'].astype(str)
        groups = df_s.groupby(strat_col)
        per_g = max(1, 1000 // max(1, len(groups)))
        picked = [grp.sample(n=min(len(grp), per_g), random_state=SEED) for _, grp in groups]
        sample_df = pd.concat(picked, ignore_index=False)
        if len(sample_df) < 1000:
            rem = df_s.drop(index=sample_df.index, errors='ignore')
            sample_df = pd.concat([sample_df, rem.sample(n=min(len(rem), 1000 - len(sample_df)), random_state=SEED)])
        elif len(sample_df) > 1000:
            sample_df = sample_df.sample(n=1000, random_state=SEED)
        sample_df = sample_df.reset_index(drop=True)
        sample_df.to_csv(GOLD_SAMPLE_PATH, index=False, encoding='utf-8-sig')
        print(f"Tersimpan: {GOLD_SAMPLE_PATH} ({len(sample_df):,} baris)")

    if not GEMINI_API_KEYS:
        print("INFO: Tidak ada GEMINI_API_KEY di Colab Secrets / .env. Langkah anotasi Gemini dilewati.")
        print("Pipeline akan tetap berjalan menggunakan gold_standard_annotated_1000.csv (jika diupload) atau rule scorer LII.")
    else:
        from google import genai
        GEMINI_MODELS = ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-2.0-flash-lite']
        print(f"Memulai anotasi Gemini ({len(sample_df):,} sampel) dengan {len(GEMINI_API_KEYS)} API key rotasi...")

        checkpoint = {}
        if os.path.exists(ANNOTATION_CKPT_PATH):
            try:
                with open(ANNOTATION_CKPT_PATH, 'r', encoding='utf-8') as f:
                    checkpoint = json.load(f)
                print(f"Melanjutkan checkpoint: {len(checkpoint)}/{len(sample_df)} selesai")
            except Exception:
                checkpoint = {}

        PROMPT_TMPL = (
            "Kamu adalah ahli linguistik Indonesia dan analis opini publik tentang Lembaga Islam Indonesia "
            "(Nahdlatul Ulama / NU, Muhammadiyah, dan BAZNAS).\n"
            "Analisislah teks komentar/berita berikut:\n"
            "Topik: {topic} | Platform: {platform}\n"
            "Teks: \"{text}\"\n\n"
            "Tentukan:\n"
            "1. sentimen: Positif / Netral / Negatif\n"
            "2. emosi: Trust / Anticipation / Joy / Disgust / Fear / Sadness / Anger / Surprise / Neutral\n"
            "3. confidence: float 0.0 - 1.0\n"
            "4. reason: penjelasan singkat <= 12 kata\n\n"
            "ATURAN: Waspadai sarkasme, pujian sinis, umpatan, kritik pengurus/zakat, serta doa baik vs tulus.\n"
            "Jawab HANYA JSON valid: "
            "{{\"sentimen\": \"...\", \"emosi\": \"...\", \"confidence\": 0.9, \"reason\": \"...\"}}"
        )

        key_idx = 0
        clients = {k: genai.Client(api_key=k) for k in GEMINI_API_KEYS}
        dead_keys = set()

        for idx, row in sample_df.iterrows():
            s_idx = str(idx)
            if s_idx in checkpoint:
                continue
            if len(dead_keys) >= len(GEMINI_API_KEYS):
                print("Semua Gemini API key kehabisan kuota sementara. Menyimpan progres terbaru...")
                break

            prompt = PROMPT_TMPL.format(
                topic=row.get('topic', 'LII'),
                platform=row.get('platform', 'SocialMedia'),
                text=str(row.get('cleaned_content', ''))[:500].replace('"', "'"),
            )
            annotated = None
            for attempt in range(len(GEMINI_API_KEYS) * len(GEMINI_MODELS)):
                cur_key = GEMINI_API_KEYS[(key_idx + attempt) % len(GEMINI_API_KEYS)]
                if cur_key in dead_keys:
                    continue
                cur_model = GEMINI_MODELS[attempt % len(GEMINI_MODELS)]
                try:
                    resp = clients[cur_key].models.generate_content(model=cur_model, contents=prompt)
                    raw_txt = (resp.text or '').strip()
                    m = re.search(r'\{.*\}', raw_txt, re.DOTALL)
                    parsed = json.loads(m.group(0) if m else raw_txt)
                    sent = str(parsed.get('sentimen', 'Netral')).strip().capitalize()
                    if sent not in SENT2ID:
                        sent = 'Netral'
                    emo = STANDARDIZE_EMO_MAP.get(str(parsed.get('emosi', 'Neutral')).strip().capitalize(), 'Neutral')
                    if emo not in EMO2ID:
                        emo = 'Neutral'
                    annotated = {
                        'gemini_sentiment': sent,
                        'gemini_emotion': emo,
                        'gemini_confidence': float(parsed.get('confidence', 0.8)),
                        'gemini_reason': str(parsed.get('reason', ''))[:200],
                    }
                    key_idx = (key_idx + attempt + 1) % len(GEMINI_API_KEYS)
                    time.sleep(0.35)
                    break
                except Exception as err:
                    err_s = str(err).upper()
                    if '429' in err_s or 'RESOURCE_EXHAUSTED' in err_s or 'QUOTA' in err_s:
                        if 'PER_DAY' in err_s or 'DAILY' in err_s:
                            dead_keys.add(cur_key)
                        time.sleep(1.5)
                    elif '401' in err_s or '403' in err_s or 'API_KEY_INVALID' in err_s:
                        dead_keys.add(cur_key)
                    else:
                        time.sleep(0.8)

            if annotated:
                checkpoint[s_idx] = annotated
                if len(checkpoint) % 20 == 0:
                    with open(ANNOTATION_CKPT_PATH, 'w', encoding='utf-8') as f:
                        json.dump(checkpoint, f, ensure_ascii=False, indent=2)
                    print(f"  Progres Anotasi: {len(checkpoint)}/{len(sample_df)} baris")

        with open(ANNOTATION_CKPT_PATH, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)

        if checkpoint:
            rows = []
            for s_idx, val in checkpoint.items():
                r_idx = int(s_idx)
                base_r = sample_df.iloc[r_idx]
                rows.append({
                    'original_index': r_idx,
                    'platform': base_r.get('platform', ''),
                    'topic': base_r.get('topic', ''),
                    'cleaned_content': base_r.get('cleaned_content', ''),
                    'old_sentiment': base_r.get('sentiment_name', ID2SENT[rule_based_sentiment_scorer_lii(base_r.get('cleaned_content', ''))]),
                    'old_emotion': base_r.get('emotion', detect_emotion_lii(base_r.get('cleaned_content', ''))),
                    'gemini_sentiment': val['gemini_sentiment'],
                    'gemini_emotion': val['gemini_emotion'],
                    'gemini_confidence': val['gemini_confidence'],
                    'gemini_reason': val['gemini_reason'],
                    'sentiment_label': SENT2ID[val['gemini_sentiment']],
                    'emotion_label': EMO2ID.get(val['gemini_emotion'], 8),
                })
            gold_df = pd.DataFrame(rows)
            gold_df.to_csv(GOLD_ANNOTATED_PATH, index=False, encoding='utf-8-sig')
            print(f"Tersimpan Gold Standard Annotated ({len(gold_df):,} baris): {GOLD_ANNOTATED_PATH}")

## CELL 4: CROSS-VALIDATE RULE-BASED vs GEMINI GOLD STANDARD (Langkah 2)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 2: CROSS-VALIDATE BASELINE RULE SCORER vs GEMINI GOLD STANDARD")
print("=" * 70)

if not gold_df.empty and 'gemini_sentiment' in gold_df.columns:
    cv_df = gold_df.dropna(subset=['cleaned_content', 'gemini_sentiment']).copy()
    if 'old_sentiment' not in cv_df.columns or cv_df['old_sentiment'].isna().all():
        cv_df['old_sentiment'] = cv_df['cleaned_content'].map(rule_based_sentiment_scorer_lii).map(ID2SENT)
    valid_cv = cv_df[cv_df['old_sentiment'].isin(SENTIMENT_LABELS) & cv_df['gemini_sentiment'].isin(SENTIMENT_LABELS)]
    if not valid_cv.empty:
        agree_pct = (valid_cv['old_sentiment'] == valid_cv['gemini_sentiment']).mean() * 100
        print(f"Sampel tervalidasi : {len(valid_cv):,} baris")
        print(f"Tingkat Agreement  : {agree_pct:.2f}%")
        print("\nConfusion Matrix (Baseline Rule Scorer vs Gemini):")
        print(pd.crosstab(valid_cv['gemini_sentiment'], valid_cv['old_sentiment'], margins=True))
    if 'gemini_emotion' in cv_df.columns:
        print("\nDistribusi 9 Emosi pada Gold Standard Gemini:")
        print(cv_df['gemini_emotion'].value_counts().to_string())
else:
    print("Gold Standard Gemini belum tersedia/lengkap. Melewati tahap cross-validation.")

## CELL 5: PREPARE & TRAIN INDOBERT 3-SENTIMEN (Langkah 3)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 3: TRAINING INDOBERT 3-SENTIMEN (NEGATIF, NETRAL, POSITIF)")
print("=" * 70)

sent_model_ready = (
    os.path.exists(os.path.join(SENT_DIR, 'config.json'))
    and (os.path.exists(os.path.join(SENT_DIR, 'model.safetensors')) or os.path.exists(os.path.join(SENT_DIR, 'pytorch_model.bin')))
)

if SKIP_IF_MODEL_EXISTS and sent_model_ready:
    print(f"SKIP_IF_MODEL_EXISTS=True & checkpoint ditemukan di {SENT_DIR}. Melewati training sentimen.")
else:
    from sklearn.model_selection import train_test_split

    train_s_path = find_lii_file('train.csv') or os.path.join(DRIVE_DIR, 'train.csv')
    val_s_path = find_lii_file('val.csv') or os.path.join(DRIVE_DIR, 'val.csv')
    test_s_path = find_lii_file('test.csv') or os.path.join(DRIVE_DIR, 'test.csv')

    def _standardize_sent_split(df_s):
        d = df_s.copy()
        if 'text' not in d.columns and 'cleaned_content' in d.columns:
            d = d.rename(columns={'cleaned_content': 'text'})
        if 'label' not in d.columns and 'sentiment_label' in d.columns:
            d = d.rename(columns={'sentiment_label': 'label'})
        d = d.dropna(subset=['text', 'label']).copy()
        d['text'] = d['text'].astype(str)
        d['label'] = pd.to_numeric(d['label'], errors='coerce').fillna(-1).astype(int)
        return d[d['label'].isin([0, 1, 2])].reset_index(drop=True)

    if os.path.exists(train_s_path) and os.path.exists(val_s_path) and os.path.exists(test_s_path):
        print("Memuat dataset sentimen pre-split dari Google Drive (train.csv / val.csv / test.csv)...")
        train_s_df = _standardize_sent_split(pd.read_csv(train_s_path, low_memory=False))
        val_s_df = _standardize_sent_split(pd.read_csv(val_s_path, low_memory=False))
        test_s_df = _standardize_sent_split(pd.read_csv(test_s_path, low_memory=False))
    else:
        print("Membangun dataset 3-Sentimen terstratifikasi (target 30.000 seimbang) dari korpus LII...")
        if master_raw_df.empty:
            master_raw_df = load_master_corpus()
        work_df = master_raw_df.copy()
        work_df['label'] = work_df['cleaned_content'].map(rule_based_sentiment_scorer_lii)

        if not gold_df.empty and 'gemini_sentiment' in gold_df.columns:
            hi = gold_df[gold_df['gemini_sentiment'].isin(SENT2ID.keys())].copy()
            if 'gemini_confidence' in hi.columns:
                hi = hi[pd.to_numeric(hi['gemini_confidence'], errors='coerce').fillna(0.8) >= 0.7]
            if not hi.empty:
                g_map = dict(zip(hi['cleaned_content'].astype(str), hi['gemini_sentiment'].map(SENT2ID)))
                ov_mask = work_df['cleaned_content'].isin(g_map.keys())
                work_df.loc[ov_mask, 'label'] = work_df.loc[ov_mask, 'cleaned_content'].map(g_map)
                print(f"Integrasi Gold Standard Gemini: {ov_mask.sum():,} baris korpus diselaraskan.")

        sampled = []
        per_cls = 10000
        for cid in [0, 1, 2]:
            sub = work_df[work_df['label'] == cid]
            if len(sub) > 0:
                sampled.append(sub.sample(n=min(len(sub), per_cls), random_state=SEED))
        bal_df = pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
        bal_df = bal_df.rename(columns={'cleaned_content': 'text'})

        train_s_df, tmp_df = train_test_split(bal_df, test_size=0.2, stratify=bal_df['label'], random_state=SEED)
        val_s_df, test_s_df = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df['label'], random_state=SEED)
        train_s_df.to_csv(train_s_path, index=False, encoding='utf-8-sig')
        val_s_df.to_csv(val_s_path, index=False, encoding='utf-8-sig')
        test_s_df.to_csv(test_s_path, index=False, encoding='utf-8-sig')

    print(f"Split Sentimen -> Train={len(train_s_df):,}, Val={len(val_s_df):,}, Test={len(test_s_df):,}")
    print("Distribusi Kelas Train:", {ID2SENT[k]: v for k, v in train_s_df['label'].value_counts().sort_index().items()})

    sent_res = train_and_evaluate_indobert(
        train_s_df, val_s_df, test_s_df,
        label_names=SENTIMENT_LABELS,
        save_dir=SENT_DIR,
        epochs=SENT_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        max_length=MAX_LENGTH,
    )

    plt.figure(figsize=(7, 6))
    sns.heatmap(sent_res['cm'], annot=True, fmt='d', cmap='Blues', xticklabels=SENTIMENT_LABELS, yticklabels=SENTIMENT_LABELS)
    plt.title(f"Confusion Matrix IndoBERT Sentimen LII (Acc: {sent_res['acc']*100:.2f}%)", fontsize=12, fontweight='bold')
    plt.xlabel("Prediksi Model")
    plt.ylabel("Aktual (Ground Truth)")
    plt.tight_layout()
    plt.savefig(os.path.join(SENT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.show()

    with open(os.path.join(SENT_DIR, 'evaluation_metrics.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'model_name': MODEL_NAME,
            'accuracy': sent_res['acc'],
            'test_accuracy': sent_res['acc'],
            'macro_precision': sent_res['prec'],
            'macro_recall': sent_res['rec'],
            'macro_f1': sent_res['mf1'],
            'weighted_f1': sent_res['wf1'],
            'confusion_matrix': sent_res['cm'].tolist(),
            'label_names': SENTIMENT_LABELS,
        }, f, indent=2)
    print(f"Metrik sentimen tersimpan ke: {SENT_DIR}/evaluation_metrics.json")

## CELL 6: PREPARE & TRAIN INDOBERT 9-EMOSI PLUTCHIK (Langkah 4)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 4: TRAINING INDOBERT 9-EMOSI (EXTENDED PLUTCHIK)")
print("  Positif : Trust, Anticipation, Joy")
print("  Negatif : Disgust, Fear, Sadness, Anger")
print("  Netral  : Surprise, Neutral")
print("=" * 70)

emo_model_ready = (
    os.path.exists(os.path.join(EMO_DIR, 'config.json'))
    and (os.path.exists(os.path.join(EMO_DIR, 'model.safetensors')) or os.path.exists(os.path.join(EMO_DIR, 'pytorch_model.bin')))
)

if SKIP_IF_MODEL_EXISTS and emo_model_ready:
    print(f"SKIP_IF_MODEL_EXISTS=True & checkpoint ditemukan di {EMO_DIR}. Melewati training 9-emosi.")
else:
    from sklearn.model_selection import train_test_split

    train_e_path = find_lii_file('train_emotion_9.csv') or os.path.join(DRIVE_DIR, 'train_emotion_9.csv')
    val_e_path = find_lii_file('val_emotion_9.csv') or os.path.join(DRIVE_DIR, 'val_emotion_9.csv')
    test_e_path = find_lii_file('test_emotion_9.csv') or os.path.join(DRIVE_DIR, 'test_emotion_9.csv')

    def _standardize_emo_split(df_e):
        d = df_e.copy()
        if 'text' not in d.columns and 'cleaned_content' in d.columns:
            d = d.rename(columns={'cleaned_content': 'text'})
        if 'label' not in d.columns and 'emotion_9' in d.columns:
            d['label'] = d['emotion_9'].replace(STANDARDIZE_EMO_MAP).map(EMO2ID)
        d = d.dropna(subset=['text', 'label']).copy()
        d['text'] = d['text'].astype(str)
        d['label'] = pd.to_numeric(d['label'], errors='coerce').fillna(-1).astype(int)
        return d[d['label'].isin(list(range(9)))].reset_index(drop=True)

    if os.path.exists(train_e_path) and os.path.exists(val_e_path) and os.path.exists(test_e_path):
        print("Memuat dataset 9-Emosi pre-split dari Google Drive (train_emotion_9.csv / val / test)...")
        train_e_df = _standardize_emo_split(pd.read_csv(train_e_path, low_memory=False))
        val_e_df = _standardize_emo_split(pd.read_csv(val_e_path, low_memory=False))
        test_e_df = _standardize_emo_split(pd.read_csv(test_e_path, low_memory=False))
    else:
        print("Membangun dataset 9-Emosi seimbang (maks 3.500/kelas) dari korpus LII...")
        if master_raw_df.empty:
            master_raw_df = load_master_corpus()
        ework = master_raw_df.copy()
        s_vals = ework['cleaned_content'].map(rule_based_sentiment_scorer_lii).map(ID2SENT)
        raw_emos = ework['cleaned_content'].map(detect_emotion_lii)
        ework['emotion_9'] = [
            align_emotion_valence_lii(t, s, e)
            for t, s, e in zip(ework['cleaned_content'], s_vals, raw_emos)
        ]

        if not gold_df.empty and 'gemini_emotion' in gold_df.columns:
            hi_e = gold_df[gold_df['gemini_emotion'].isin(EMO2ID.keys())].copy()
            if 'gemini_confidence' in hi_e.columns:
                hi_e = hi_e[pd.to_numeric(hi_e['gemini_confidence'], errors='coerce').fillna(0.8) >= 0.7]
            if not hi_e.empty:
                e_map = dict(zip(hi_e['cleaned_content'].astype(str), hi_e['gemini_emotion']))
                ov = ework['cleaned_content'].isin(e_map.keys())
                ework.loc[ov, 'emotion_9'] = ework.loc[ov, 'cleaned_content'].map(e_map)

        ework['label'] = ework['emotion_9'].map(EMO2ID)
        ework = ework.dropna(subset=['label']).copy()
        ework['label'] = ework['label'].astype(int)

        samples = []
        for emo in EMOTION_9_CLASSES:
            sub = ework[ework['emotion_9'] == emo]
            if len(sub) == 0:
                continue
            n_take = min(3500, len(sub))
            max_plat = max(1, int(n_take * 0.45))
            parts = []
            for plat in sub['platform'].unique():
                pm = sub[sub['platform'] == plat]
                parts.append(pm.sample(n=min(len(pm), max_plat), random_state=SEED))
            comb = pd.concat(parts) if parts else sub
            if len(comb) < n_take:
                rem = sub.drop(index=comb.index, errors='ignore')
                comb = pd.concat([comb, rem.sample(n=min(len(rem), n_take - len(comb)), random_state=SEED)])
            elif len(comb) > n_take:
                comb = comb.sample(n=n_take, random_state=SEED)
            samples.append(comb)

        ebal = pd.concat(samples, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
        ebal = ebal.rename(columns={'cleaned_content': 'text'})

        train_e_df, etmp = train_test_split(ebal, test_size=0.2, stratify=ebal['label'], random_state=SEED)
        val_e_df, test_e_df = train_test_split(etmp, test_size=0.5, stratify=etmp['label'], random_state=SEED)
        train_e_df.to_csv(train_e_path, index=False, encoding='utf-8-sig')
        val_e_df.to_csv(val_e_path, index=False, encoding='utf-8-sig')
        test_e_df.to_csv(test_e_path, index=False, encoding='utf-8-sig')

    print(f"Split 9-Emosi -> Train={len(train_e_df):,}, Val={len(val_e_df):,}, Test={len(test_e_df):,}")

    emo_res = train_and_evaluate_indobert(
        train_e_df, val_e_df, test_e_df,
        label_names=EMOTION_9_CLASSES,
        save_dir=EMO_DIR,
        epochs=EMO_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        max_length=MAX_LENGTH,
    )

    pred_sent_impl = [EMOTION_TO_SENTIMENT[ID2EMO[p]] for p in emo_res['test_preds']]
    true_sent_impl = [EMOTION_TO_SENTIMENT[ID2EMO[t]] for t in emo_res['test_trues']]
    sent_implied_acc = accuracy_score(true_sent_impl, pred_sent_impl)
    print(f"Konsistensi Sentimen Implisit dari 9 Emosi: {sent_implied_acc*100:.2f}%")

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(emo_res['cm'], annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_9_CLASSES, yticklabels=EMOTION_9_CLASSES, ax=ax)
    ax.set_title(f"Confusion Matrix IndoBERT 9 Emosi LII (Test Acc = {emo_res['acc']*100:.2f}%)", fontsize=13, fontweight='bold')
    ax.set_xlabel("Predicted Emotion")
    ax.set_ylabel("Actual Emotion")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(EMO_DIR, 'confusion_matrix_9_emotions.png'), dpi=300)
    plt.show()

    with open(os.path.join(EMO_DIR, 'evaluation_metrics.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'model_name': MODEL_NAME,
            'task': 'emotion_classification_9_classes',
            'classes': EMOTION_9_CLASSES,
            'label_names': EMOTION_9_CLASSES,
            'label2id': emo_res['label2id'],
            'id2label': emo_res['id2label'],
            'emotion_to_sentiment': EMOTION_TO_SENTIMENT,
            'accuracy': emo_res['acc'],
            'test_accuracy': emo_res['acc'],
            'macro_precision': emo_res['prec'],
            'macro_recall': emo_res['rec'],
            'macro_f1': emo_res['mf1'],
            'weighted_f1': emo_res['wf1'],
            'implicit_sentiment_accuracy': float(sent_implied_acc),
            'confusion_matrix': emo_res['cm'].tolist(),
        }, f, indent=2)
    print(f"Metrik 9-Emosi tersimpan ke: {EMO_DIR}/evaluation_metrics.json")

## CELL 7: FULL CORPUS GPU T4 INFERENCE (Langkah 5)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 5: INFERENSI FULL CORPUS LII DENGAN GPU T4")
print("=" * 70)

# Set True HANYA jika sengaja ingin memakai enriched_corpus_for_sna lama saat master_scraped_corpus belum diupload
ALLOW_ENRICHED_FALLBACK = False


def _resolve_corpus_for_infer():
    """Cari master_scraped_corpus terbaru (refresh FUSE Drive otomatis jika baru diupload)."""
    def _scan(fname):
        cands = [
            os.path.join(DRIVE_DIR, fname),
            os.path.join(DRIVE_DIR, 'data', fname),
            f'/content/drive/MyDrive/IndoBERT_SNA_MT_LII/data/{fname}',
            f'/content/drive/MyDrive/IndoBERT_SNA_MT_LII/{fname}',
            f'/content/drive/MyDrive/{fname}',
            f'/content/{fname}',
            f'/content/data/{fname}',
        ]
        for p in cands:
            if os.path.exists(p):
                return p
        return None

    pq = _scan('master_scraped_corpus.parquet')
    csv_p = _scan('master_scraped_corpus.csv')

    # Jika baru saja diupload ke Google Drive, paksa remount ringan agar cache FUSE Colab langsung sinkron
    if not pq and not csv_p:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=True)
        except Exception:
            pass
        pq = _scan('master_scraped_corpus.parquet')
        csv_p = _scan('master_scraped_corpus.csv')

    if pq:
        print(f"Memuat korpus utama (3 topik lengkap) dari Parquet: {pq}")
        df = pd.read_parquet(pq)
    elif csv_p:
        print(f"Memuat korpus utama (3 topik lengkap) dari CSV: {csv_p}")
        df = pd.read_csv(csv_p, low_memory=False)
    elif ALLOW_ENRICHED_FALLBACK and os.path.exists(ENRICHED_PQ_PATH):
        print(f"PERINGATAN: Memakai fallback {ENRICHED_PQ_PATH} (cek apakah topik Muhammadiyah sudah termasuk!).")
        df = pd.read_parquet(ENRICHED_PQ_PATH)
    else:
        isi = sorted(os.listdir(DRIVE_DIR)) if os.path.exists(DRIVE_DIR) else []
        extra_note = ""
        if os.path.exists(ENRICHED_PQ_PATH):
            try:
                old_topics = pd.read_parquet(ENRICHED_PQ_PATH, columns=['topic'])['topic'].value_counts().to_dict()
                extra_note = (
                    f"\nPERHATIAN: Di Drive ada enriched_corpus_for_sna.parquet lama dengan topik {old_topics} "
                    "(belum memuat 138.756 baris lengkap NU + Muhammadiyah + BAZNAS)."
                )
            except Exception:
                pass
        raise FileNotFoundError(
            f"master_scraped_corpus.parquet / .csv belum ditemukan di {DRIVE_DIR}!\n"
            f"Isi folder Drive saat ini: {isi}{extra_note}\n"
            "Solusi: Upload file lokal data/master_scraped_corpus.parquet (78.6 MB) ke MyDrive/IndoBERT_SNA_LII/, "
            "lalu jalankan ulang Cell 7 ini (tidak perlu ulang training Cell 5 & Cell 6)."
        )

    if 'content' in df.columns and 'cleaned_content' not in df.columns:
        df['cleaned_content'] = df['content']
    df['cleaned_content'] = df['cleaned_content'].fillna('').astype(str)
    df = df[df['cleaned_content'].str.strip().str.len() > 8].reset_index(drop=True)
    df['topic'] = normalize_lii_topics(df)
    return df


if not FORCE_INFER and os.path.exists(ENRICHED_PQ_PATH):
    print(f"FORCE_INFER=False & file sudah ada -> Memuat {ENRICHED_PQ_PATH}")
    master = pd.read_parquet(ENRICHED_PQ_PATH)
else:
    if 'master_raw_df' in globals() and isinstance(master_raw_df, pd.DataFrame) and not master_raw_df.empty:
        master = master_raw_df.copy()
    else:
        master_raw_df = _resolve_corpus_for_infer()
        master = master_raw_df.copy()
    print(f"Total korpus LII yang akan di-inferensi: {len(master):,} baris")
    print("Rincian Topik Korpus:", master['topic'].value_counts().to_dict())

    s_model = AutoModelForSequenceClassification.from_pretrained(SENT_DIR).to(device)
    s_model.eval()
    s_tok = AutoTokenizer.from_pretrained(SENT_DIR)

    e_model = AutoModelForSequenceClassification.from_pretrained(EMO_DIR).to(device)
    e_model.eval()
    e_tok = AutoTokenizer.from_pretrained(EMO_DIR)

    def batch_infer_texts(model, tok, texts, bs=128, ml=128, label_tag="Model"):
        preds, probs = [], []
        n = len(texts)
        for i in range(0, n, bs):
            bt = [normalize_indobert_text(t)[:500] for t in texts[i:i + bs]]
            enc = tok(bt, truncation=True, padding=True, max_length=ml, return_tensors='pt').to(device)
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=use_cuda):
                    logits = model(**enc).logits
                pb = torch.softmax(logits.float(), dim=1).cpu().numpy()
            preds.extend(np.argmax(pb, axis=1))
            probs.append(pb)
            if (i // bs + 1) % 100 == 0 or (i + bs) >= n:
                done = min(i + bs, n)
                print(f"  [{label_tag}] Inferensi: {done:,}/{n:,} ({done/n*100:.1f}%)")
        return np.array(preds), np.vstack(probs)

    raw_texts = master['cleaned_content'].values
    print("\nMenjalankan Inferensi IndoBERT 3-Sentimen...")
    sp, sprob = batch_infer_texts(s_model, s_tok, raw_texts, bs=128, ml=MAX_LENGTH, label_tag="Sentimen")
    master['sentiment_label'] = sp.astype('int8')
    master['predicted_sentiment'] = pd.Series(sp).map(ID2SENT).fillna('Netral').values
    master['sentiment_name'] = master['predicted_sentiment']
    master['sentiment_score'] = (sprob[:, 2] - sprob[:, 0]).astype('float32')

    print("\nMenjalankan Inferensi IndoBERT 9-Emosi...")
    epred, eprob = batch_infer_texts(e_model, e_tok, raw_texts, bs=128, ml=MAX_LENGTH, label_tag="9-Emosi")
    raw_id2emo = (
        {int(k): v for k, v in e_model.config.id2label.items()}
        if getattr(e_model.config, 'id2label', None)
        else ID2EMO
    )
    raw_emo_names = [raw_id2emo.get(int(p), 'Neutral') for p in epred]
    aligned_emos = [
        align_emotion_valence_lii(txt, snt, emo)
        for txt, snt, emo in zip(raw_texts, master['predicted_sentiment'].values, raw_emo_names)
    ]
    master['emotion'] = aligned_emos
    master['emotion_9'] = aligned_emos
    master['emotion_confidence'] = np.max(eprob, axis=1).astype('float32')
    master['emotion_sentiment'] = master['emotion_9'].map(EMOTION_TO_SENTIMENT)

    # Sinkronisasi kolom tanggal & metrik engagement untuk Streamlit & SNA
    if 'date' not in master.columns or master['date'].isna().all():
        if 'created_at' in master.columns:
            master['date'] = pd.to_datetime(
                master['created_at'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0], errors='coerce'
            )
        else:
            master['date'] = pd.NaT
    master['date'] = pd.to_datetime(master['date'], errors='coerce')
    if master['date'].isna().any():
        med_dt = master['date'].dropna().median()
        master['date'] = master['date'].fillna(med_dt if pd.notna(med_dt) else pd.Timestamp.now().normalize())
    master['date_str'] = master['date'].dt.strftime('%Y-%m-%d')
    master['year_month'] = master['date'].dt.to_period('M').astype(str)

    for c in ['likes', 'comments', 'shares', 'reposts', 'views']:
        master[c] = pd.to_numeric(master[c], errors='coerce').fillna(0).clip(0, 500000).astype('int32') if c in master.columns else 0
    master['total_engagement'] = (master['likes'] + master['comments'] + master['shares'] + master['reposts']).astype('int32')
    master['reach'] = (master['views'] + master['likes'] * 3 + master['comments'] * 5 + master['shares'] * 10 + master['reposts'] * 10).astype('int32')

    master.to_parquet(ENRICHED_PQ_PATH, index=False)
    master.to_csv(ENRICHED_CSV_PATH, index=False, encoding='utf-8-sig')
    print(f"Tersimpan Parquet : {ENRICHED_PQ_PATH}")
    print(f"Tersimpan CSV     : {ENRICHED_CSV_PATH}")

print("\nDistribusi Sentimen LII:")
print(master['predicted_sentiment'].value_counts().to_string())
print("\nDistribusi 9 Emosi LII:")
print(master['emotion_9'].value_counts().to_string())
print("\nDistribusi per Topik LII:")
print(pd.crosstab(master['topic'], master['predicted_sentiment'], margins=True))

## CELL 8: BUILD SOCIAL NETWORK ANALYSIS (SNA) BY SENTIMENT & EMOTION (Langkah 6)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 6: MEMBANGUN GRAF SNA BY SENTIMENT & EMOTION (LII)")
print("=" * 70)

import os
import re
import json
import pandas as pd
import networkx as nx
from collections import defaultdict, Counter

if 'master' not in globals() or master is None or len(master) == 0:
    print(f"Memuat korpus hasil inferensi dari: {ENRICHED_PQ_PATH}")
    master = pd.read_parquet(ENRICHED_PQ_PATH)

os.makedirs(SNA_OUT_DIR, exist_ok=True)
MENTION_RE = re.compile(r'@([A-Za-z0-9_\.]{2,40})')
SENTIMENT_COLORS = {'Positif': '#10B981', 'Netral': '#64748B', 'Negatif': '#EF4444'}


def build_and_export_sna_lii(df_corpus, topic_name='All_Topics', min_interactions=1):
    if topic_name != 'All_Topics':
        df_sub = df_corpus[df_corpus['topic'].astype(str).str.lower() == topic_name.lower()].copy()
    else:
        df_sub = df_corpus.copy()

    print(f"\n--- Graf SNA Topik: {topic_name} ({len(df_sub):,} baris) ---")
    if df_sub.empty:
        print(f"Tidak ada data untuk topik {topic_name}, dilewati.")
        return None

    G = nx.DiGraph()
    author_sentiments = defaultdict(lambda: {'Positif': 0, 'Netral': 0, 'Negatif': 0})
    author_emotions = defaultdict(lambda: defaultdict(int))
    author_platforms = {}
    author_engagements = defaultdict(int)
    target_sentiments = defaultdict(lambda: {'Positif': 0, 'Netral': 0, 'Negatif': 0})
    target_emotions = defaultdict(lambda: defaultdict(int))
    edges_dict = defaultdict(lambda: {'weight': 0, 'sentiments': [], 'emotions': []})
    hashtag_authors = defaultdict(list)

    for row in df_sub.to_dict('records'):
        src_raw = str(row.get('handle') or row.get('author') or row.get('user_id') or '').strip().lstrip('@').lower()
        if not src_raw or src_raw in ('nan', 'none', 'anonim'):
            src_raw = f"{str(row.get('platform', 'web')).lower()}_user"

        sent = str(row.get('sentiment_name') or row.get('predicted_sentiment') or 'Netral')
        if sent not in ('Positif', 'Netral', 'Negatif'):
            sent = 'Netral'
        emo = str(row.get('emotion_9') or row.get('emotion') or 'Neutral')
        eng_val = row.get('total_engagement', 0)
        eng = int(eng_val) if pd.notna(eng_val) else 0
        plat = str(row.get('platform', 'SocialMedia'))

        author_sentiments[src_raw][sent] += 1
        author_emotions[src_raw][emo] += 1
        author_platforms[src_raw] = plat
        author_engagements[src_raw] += eng

        targets = set()
        m_col = str(row.get('mentions') or '').strip()
        if m_col and m_col.lower() != 'nan':
            for m in m_col.split(','):
                mc = m.strip().lstrip('@').lower()
                if len(mc) >= 2:
                    targets.add(mc)
        raw_txt = str(row.get('content') or row.get('raw_content') or row.get('cleaned_content') or '')
        for m in MENTION_RE.findall(raw_txt):
            targets.add(m.lower())

        for tgt in targets:
            if tgt and tgt != src_raw:
                target_sentiments[tgt][sent] += 1
                target_emotions[tgt][emo] += 1
                ek = (src_raw, tgt)
                edges_dict[ek]['weight'] += 1
                edges_dict[ek]['sentiments'].append(sent)
                edges_dict[ek]['emotions'].append(emo)

        h_col = str(row.get('hashtags') or '').strip()
        if h_col and h_col.lower() != 'nan':
            for h in h_col.split():
                hc = h.strip().lower()
                if len(hc) >= 4:
                    hashtag_authors[hc].append((src_raw, sent, emo))

    for _, entries in hashtag_authors.items():
        uniq_users = []
        seen_u = set()
        for u, s, e in entries:
            if u not in seen_u:
                seen_u.add(u)
                uniq_users.append((u, s, e))
        if 2 <= len(uniq_users) <= 25:
            hub_user, _, _ = uniq_users[0]
            for u, s, e in uniq_users[1:]:
                ek = (u, hub_user)
                edges_dict[ek]['weight'] += 1
                edges_dict[ek]['sentiments'].append(s)
                edges_dict[ek]['emotions'].append(e)

    all_users = set(author_sentiments.keys()).union(set(target_sentiments.keys()))
    for u in all_users:
        s_cnt = author_sentiments[u] if sum(author_sentiments[u].values()) > 0 else target_sentiments[u]
        dom_sent = max(s_cnt, key=s_cnt.get) if s_cnt else 'Netral'
        e_cnt = author_emotions[u] if sum(author_emotions[u].values()) > 0 else target_emotions[u]
        dom_emo = max(e_cnt, key=e_cnt.get) if e_cnt else 'Neutral'
        tot_s = sum(s_cnt.values())
        actor_nss = round(((s_cnt.get('Positif', 0) - s_cnt.get('Negatif', 0)) / tot_s * 100), 2) if tot_s > 0 else 0.0
        G.add_node(
            u,
            platform=author_platforms.get(u, 'Mentioned'),
            sentiment=dom_sent,
            emotion=dom_emo,
            nss=actor_nss,
            total_engagement=int(author_engagements.get(u, 0)),
            color=SENTIMENT_COLORS.get(dom_sent, '#64748B'),
        )

    for (src, tgt), edata in edges_dict.items():
        if edata['weight'] >= min_interactions:
            most_sent = Counter(edata['sentiments']).most_common(1)[0][0]
            most_emo = Counter(edata['emotions']).most_common(1)[0][0]
            G.add_edge(src, tgt, weight=int(edata['weight']), sentiment=most_sent, emotion=most_emo)

    in_deg = dict(G.in_degree(weight='weight'))
    out_deg = dict(G.out_degree(weight='weight'))
    nx.set_node_attributes(G, in_deg, 'in_degree')
    nx.set_node_attributes(G, out_deg, 'out_degree')

    top_inf = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:20]
    print(f"Graf SNA dibangun: {G.number_of_nodes():,} Nodes, {G.number_of_edges():,} Edges")
    top_summary = []
    for rank, (u, deg) in enumerate(top_inf, 1):
        nd = G.nodes[u]
        if rank <= 10:
            print(f"  {rank:>2}. @{u:<24} InDeg={deg:>5} | {nd['sentiment']:<8} | {nd['emotion']:<12} | NSS={nd['nss']:>6.1f}%")
        top_summary.append({
            'rank': rank, 'user': u, 'in_degree': deg, 'out_degree': out_deg.get(u, 0),
            'sentiment': nd['sentiment'], 'emotion': nd['emotion'], 'nss': nd['nss'],
            'platform': nd['platform'], 'engagement': nd['total_engagement'],
        })

    prefix = topic_name.lower()
    gexf_file = os.path.join(SNA_OUT_DIR, f"{prefix}_sna_by_sentiment.gexf")
    json_file = os.path.join(SNA_OUT_DIR, f"{prefix}_sna_summary.json")
    nx.write_gexf(G, gexf_file)
    with open(json_file, 'w', encoding='utf-8') as fj:
        json.dump({
            'topic': topic_name,
            'total_nodes': G.number_of_nodes(),
            'total_edges': G.number_of_edges(),
            'top_influencers': top_summary,
        }, fj, indent=2)
    print(f"Ekspor Gephi & JSON: {gexf_file}")
    return G


build_and_export_sna_lii(master, 'All_Topics')
for top in LII_TOPICS:
    if (master['topic'].astype(str).str.lower() == top.lower()).sum() > 0:
        build_and_export_sna_lii(master, top)

## CELL 9: BACKUP DELIVERABLES KE ZIP (TANPA .ENV) & DOWNLOAD (Langkah 7)

In [ ]:
print("\n" + "=" * 70)
print("LANGKAH 7: BACKUP HASIL TRAINING, INFERENSI & SNA LII KE ZIP")
print("=" * 70)

import os
import shutil
import zipfile
from google.colab import files

TEMP_ZIP = '/content/hasil_training_indobert_sna_LII.zip'
DRIVE_ZIP = os.path.join(DRIVE_DIR, 'hasil_training_indobert_sna_LII.zip')

# Kumpulkan hanya folder/file deliverable agar file .env atau zip lama tidak ikut terkemas
DELIVERABLES = [
    'best_indobert_sentiment',
    'best_indobert_emotion',
    'enriched_corpus_for_sna.parquet',
    'enriched_corpus_for_sna.csv',
    'sna_output',
    'gold_standard_annotated_1000.csv',
]

STORED_EXTS = ('.safetensors', '.bin', '.pt', '.parquet', '.png')

added_count = 0
with zipfile.ZipFile(TEMP_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for item in DELIVERABLES:
        src_path = os.path.join(DRIVE_DIR, item)
        if not os.path.exists(src_path):
            print(f"  [Lewati - belum ada]: {item}")
            continue
        print(f"  Mengemas: {item} ...")
        if os.path.isdir(src_path):
            for root, _, fnames in os.walk(src_path):
                for fn in fnames:
                    full_p = os.path.join(root, fn)
                    rel_p = os.path.relpath(full_p, DRIVE_DIR)
                    ctype = zipfile.ZIP_STORED if fn.lower().endswith(STORED_EXTS) else zipfile.ZIP_DEFLATED
                    zf.write(full_p, rel_p, compress_type=ctype)
                    added_count += 1
        else:
            ctype = zipfile.ZIP_STORED if item.lower().endswith(STORED_EXTS) else zipfile.ZIP_DEFLATED
            zf.write(src_path, item, compress_type=ctype)
            added_count += 1

shutil.copy2(TEMP_ZIP, DRIVE_ZIP)
zip_mb = os.path.getsize(DRIVE_ZIP) / (1024 * 1024)
print(f"\nTotal {added_count:,} file berhasil dikemas ke: {DRIVE_ZIP} ({zip_mb:.1f} MB)")
print("Catatan: File sensitif seperti .env tidak dimasukkan ke dalam arsip ZIP.")
print("Tip Cepat: Jika popup download browser lambat karena ukuran ~1 GB, unduh langsung dari Google Drive (Klik kanan hasil_training_indobert_sna_LII.zip -> Download).")

try:
    files.download(DRIVE_ZIP)
except Exception as e:
    print(f"Download browser otomatis tidak aktif ({e}). Silakan unduh langsung dari Google Drive: {DRIVE_ZIP}")